# OCR Reality-Check Spike

<!-- To add real photos later: drop image files (.png/.jpg/.jpeg/.tif) into `data/samples/` and re-run the notebook top to bottom. No code changes needed. -->

A self-contained Google Colab notebook that compares **Tesseract** and **EasyOCR** on a
synthetic "ledger" image. No external data is required — the notebook generates its own
test image. Runnable top to bottom with no manual steps.

**How to add real photos later:** drop any `.png`/`.jpg`/`.jpeg`/`.tif` files into
`data/samples/` and re-run. They will be picked up automatically alongside the synthetic image.

## 1. Install dependencies

Installs the Python OCR stack and the Tesseract engine binary.

In [ ]:
# Python OCR + imaging libraries
!pip install -q pytesseract easyocr opencv-python pillow

# Tesseract engine binary (needed by pytesseract). Safe no-op if already present.
!apt-get -qq update && apt-get -qq install -y tesseract-ocr

## 2. Imports and paths

In [ ]:
import os
import glob

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# Directory where sample images live (synthetic + any real photos added later).
SAMPLES_DIR = "data/samples"
os.makedirs(SAMPLES_DIR, exist_ok=True)

SYNTHETIC_PATH = os.path.join(SAMPLES_DIR, "synthetic_ledger.png")
print("Samples directory:", os.path.abspath(SAMPLES_DIR))

## 3. Generate a synthetic "ledger" image

Builds a ~6-row table (date, item, quantity, price, total) with PIL. Uses a
handwriting-style font if one is available on the machine, otherwise falls back to the
default bitmap font. Mild Gaussian noise and a slight rotation are added to mimic a phone
photo of a real handwritten ledger.

In [ ]:
def _load_font(size):
    """Try a handwriting-style font, then any TrueType font, else PIL's default bitmap font."""
    candidates = [
        # Handwriting-ish fonts sometimes present in Colab / Linux images.
        "/usr/share/fonts/truetype/humor-sans/Humor-Sans.ttf",
        "Humor-Sans.ttf",
        "/usr/share/fonts/truetype/comic-sans/ComicSans.ttf",
        # Generic fallbacks that are usually installed.
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
        "DejaVuSans.ttf",
    ]
    for path in candidates:
        try:
            font = ImageFont.truetype(path, size)
            print(f"Using font: {path}")
            return font
        except (OSError, IOError):
            continue
    print("No TrueType font found — falling back to PIL default bitmap font.")
    return ImageFont.load_default()


def generate_synthetic_ledger(out_path, seed=42):
    """Render a small ledger table, add noise + rotation, and save to out_path."""
    rng = np.random.default_rng(seed)

    rows = [
        ["Date", "Item", "Qty", "Price", "Total"],
        ["2026-01-03", "Maize meal", "2", "3.50", "7.00"],
        ["2026-01-05", "Cooking oil", "1", "4.25", "4.25"],
        ["2026-01-09", "Sugar 2kg", "3", "2.10", "6.30"],
        ["2026-01-12", "Bread", "5", "1.15", "5.75"],
        ["2026-01-15", "Soap bar", "4", "0.90", "3.60"],
        ["2026-01-20", "Tea leaves", "1", "2.75", "2.75"],
    ]

    # Canvas.
    W, H = 900, 460
    img = Image.new("RGB", (W, H), color=(252, 250, 244))  # off-white paper tone
    draw = ImageDraw.Draw(img)

    font = _load_font(size=26)
    header_font = _load_font(size=28)

    # Column x-offsets and row spacing.
    col_x = [30, 210, 470, 580, 720]
    y0, row_h = 40, 56

    # Title.
    draw.text((30, 8), "Shop Ledger — January", fill=(20, 20, 20), font=header_font)

    for r, row in enumerate(rows):
        y = y0 + r * row_h
        this_font = header_font if r == 0 else font
        for c, cell in enumerate(row):
            draw.text((col_x[c], y), str(cell), fill=(15, 15, 15), font=this_font)
        # Ruled line under each row.
        draw.line([(20, y + row_h - 12), (W - 20, y + row_h - 12)], fill=(200, 195, 180), width=1)

    # Convert to array for noise.
    arr = np.asarray(img).astype(np.float32)
    noise = rng.normal(0, 8.0, arr.shape)  # mild Gaussian noise
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr)

    # Slight rotation to mimic a hand-held phone photo (expand keeps corners).
    img = img.rotate(-3.0, resample=Image.BICUBIC, expand=True, fillcolor=(252, 250, 244))

    img.save(out_path)
    print(f"Saved synthetic ledger to: {out_path}  ({img.size[0]}x{img.size[1]})")
    return out_path


generate_synthetic_ledger(SYNTHETIC_PATH)

## 4. Collect all sample images

Picks up the synthetic image plus any real photos already dropped into `data/samples/`.

In [ ]:
def collect_images(samples_dir):
    """Return a sorted, de-duplicated list of image paths in samples_dir."""
    exts = ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff", "*.bmp")
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(samples_dir, ext)))
        paths.extend(glob.glob(os.path.join(samples_dir, ext.upper())))
    return sorted(set(paths))


image_paths = collect_images(SAMPLES_DIR)
print(f"Found {len(image_paths)} image(s):")
for p in image_paths:
    tag = "(synthetic)" if os.path.abspath(p) == os.path.abspath(SYNTHETIC_PATH) else "(real)"
    print(f"  - {p} {tag}")

## 5. OCR engines

`run_tesseract` and `run_easyocr` each take an image path and return recognised text as a
single string. The EasyOCR reader is built once and cached, since model download/init is
slow.

In [ ]:
import pytesseract
import easyocr

# Build the EasyOCR reader once (downloads models on first use). CPU is fine for Colab.
_EASYOCR_READER = None


def _get_easyocr_reader():
    global _EASYOCR_READER
    if _EASYOCR_READER is None:
        _EASYOCR_READER = easyocr.Reader(["en"], gpu=False)
    return _EASYOCR_READER


def run_tesseract(img_path):
    """Return the text Tesseract recognises in the image at img_path."""
    img = Image.open(img_path)
    return pytesseract.image_to_string(img).strip()


def run_easyocr(img_path):
    """Return the text EasyOCR recognises in the image at img_path (one line per detection)."""
    reader = _get_easyocr_reader()
    results = reader.readtext(img_path, detail=0, paragraph=False)
    return "\n".join(results).strip()

## 6. Run OCR on every image and compare side by side

For each image: display it, then print Tesseract output next to EasyOCR output under clear
headings.

In [ ]:
def _print_side_by_side(left_title, left_text, right_title, right_text, width=44):
    """Print two blocks of text in aligned columns."""
    left_lines = (left_text or "(no text)").splitlines() or ["(no text)"]
    right_lines = (right_text or "(no text)").splitlines() or ["(no text)"]
    n = max(len(left_lines), len(right_lines))
    left_lines += [""] * (n - len(left_lines))
    right_lines += [""] * (n - len(right_lines))

    header = f"{left_title:<{width}} | {right_title}"
    print(header)
    print("-" * len(header))
    for l, r in zip(left_lines, right_lines):
        print(f"{l[:width]:<{width}} | {r}")


for path in image_paths:
    print("=" * 100)
    print(f"IMAGE: {path}")
    print("=" * 100)

    # Display the image.
    img = Image.open(path)
    plt.figure(figsize=(9, 5))
    plt.imshow(img)
    plt.axis("off")
    plt.title(os.path.basename(path))
    plt.show()

    # Run both engines.
    tess_text = run_tesseract(path)
    easy_text = run_easyocr(path)

    _print_side_by_side("TESSERACT", tess_text, "EASYOCR", easy_text)
    print()

## 7. What to look for

This is a quick reality-check, not a benchmark. When reading the side-by-side output above,
compare the two engines on the following:

- **Which engine read more correctly?** Count how many of the ledger cells (dates, item
  names, quantities, prices, totals) each engine reproduced accurately. On clean, upright
  printed text Tesseract is often competitive; on rotated, noisy, or handwriting-style
  images EasyOCR (deep-learning based) usually degrades more gracefully.
- **Numbers vs. words.** Ledgers live or die on the digits. Watch for classic confusions:
  `0`↔`O`, `1`↔`l`↔`I`, `5`↔`S`, `.`↔`,` in prices. An engine that reads the words but
  garbles the totals is useless for a ledger.
- **Layout / row structure.** Tesseract returns a single flowing string and can merge or
  reorder columns; EasyOCR returns per-detection boxes (one per line here) and may preserve
  the row grouping better — or split a single row into several fragments.
- **Where each failed.** Note whether errors cluster around the rotation (skewed baselines),
  the ruled lines (mistaken for characters), or the noise. This tells you what preprocessing
  (deskew, denoise, binarisation) would help most before committing to an engine.
- **Missing rows.** Did either engine drop a row entirely? A silent omission is worse than a
  visible misread for downstream bookkeeping.

**Next step:** add a couple of real phone photos to `data/samples/` and re-run. The
synthetic image tells you the engines work; real photos tell you whether they work *for your
data*.